# 📊 ACE-Net Dataset Preprocessing Master Dashboard
### Real-Time Live Audit across all 14,815 Clips, 22 Shards & 6 Datasets

I-mount lamang ang Google Drive at i-run ang mga cells sa ibaba para makita agad kung:
1. **Ilang clips na ang tapos** bawat dataset at sa buong 14,815 dataset.
2. **Progress bar (%)** ng bawat shard mula Shard 1 hanggang Shard 22.
3. **Integrity audit** ng mga na-generate na `.npy` at `.jpg` feature tensors sa Google Drive.

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os, sys

drive.mount('/content/drive')
print('Google Drive connected successfully!')

## Step 2: Run Real-Time Progress & Tensor Audit Dashboard

In [ ]:
import json, glob, csv
from pathlib import Path
import pandas as pd

DRIVE_ROOT = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline preprocessed')

# Master Ground-Truth Targets (Total: 14,815 clips across 22 shards)
DATASET_TARGETS = {
    'CMU-MOSEI': {'shards': 7, 'total_clips': 5020},
    'MELD':      {'shards': 5, 'total_clips': 3234},
    'TRACK_1':   {'shards': 2, 'total_clips': 1164},
    'MUSTARD':   {'shards': 1, 'total_clips': 595},
    'TRACK_2':   {'shards': 3, 'total_clips': 1818},
    'TRACK_3':   {'shards': 4, 'total_clips': 2984},
}

GRAND_TOTAL_DATASET = 14815

if not DRIVE_ROOT.exists():
    print(f"[ERROR] Path not found: {DRIVE_ROOT}")
    print("Siguraduhin na may shortcut ng 'THESIS_MOTHERFILE' sa 'My Drive'!")
else:
    print("=" * 85)
    print("        ACE-NET PREPROCESSING LIVE DASHBOARD (REAL-TIME AUDIT)")
    print("=" * 85)
    
    overall_completed = 0
    overall_failed = 0
    overall_audios = 0
    overall_texts = 0
    overall_visuals = 0
    
    summary_rows = []
    
    for dname, info in DATASET_TARGETS.items():
        d_dir = DRIVE_ROOT / dname
        n_shards = info['shards']
        d_target_clips = info['total_clips']
        
        d_comp = 0
        d_fail = 0
        d_aud = 0
        d_txt = 0
        d_vis = 0
        
        print(f"\n📁 [{dname}] (Target: {d_target_clips:,} clips across {n_shards} shards)")
        print("-" * 85)
        print(f"  {'Shard ID':<12} | {'Status':<24} | {'Completed':<10} | {'Failed':<8} | {'Audio Tensors':<14} | {'Visual Folders'}")
        print("  " + "-" * 81)
        
        for s_idx in range(1, n_shards + 1):
            shard_name = f"shard_{s_idx:04d}"
            s_dir = d_dir / "shards" / shard_name
            ckpt_path = d_dir / "checkpoints" / f"{shard_name}_checkpoint.json"
            
            status = "NOT_STARTED"
            s_comp = 0
            s_fail = 0
            
            # Read Checkpoint if exists
            if ckpt_path.exists():
                try:
                    with open(ckpt_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        status = data.get('status', 'IN_PROGRESS')
                        s_comp = len(data.get('completed_ids', []))
                        s_fail = len(data.get('failed_ids', []))
                except Exception:
                    status = "READ_ERROR"
            
            # Count actual tensors on disk
            aud_count = len(list((s_dir / 'audio').glob('*.npy'))) if (s_dir / 'audio').exists() else 0
            txt_count = len(list((s_dir / 'text').glob('*_input_ids.npy'))) if (s_dir / 'text').exists() else 0
            vis_count = len([x for x in (s_dir / 'visual').iterdir() if x.is_dir()]) if (s_dir / 'visual').exists() else 0
            
            # Fallback if checkpoint was interrupted but tensors exist on disk
            if s_comp == 0 and aud_count > 0:
                s_comp = aud_count
                status = "PROCESSING (DISK)"
                
            d_comp += s_comp
            d_fail += s_fail
            d_aud += aud_count
            d_txt += txt_count
            d_vis += vis_count
            
            # Print individual shard row
            status_disp = f"🟢 {status}" if "COMPLETED" in status else (f"🟡 {status}" if s_comp > 0 else f"⚪ {status}")
            print(f"  {shard_name:<12} | {status_disp:<24} | {s_comp:>9,} | {s_fail:>7,} | {aud_count:>13,} | {vis_count:>13,}")
            
        overall_completed += d_comp
        overall_failed += d_fail
        overall_audios += d_aud
        overall_texts += d_txt
        overall_visuals += d_vis
        
        pct = (d_comp / d_target_clips) * 100 if d_target_clips > 0 else 0
        print("  " + "-" * 81)
        print(f"  👉 {dname} Subtotal: {d_comp:,} / {d_target_clips:,} clips processed ({pct:.1f}%)")
        
        summary_rows.append({
            'Dataset': dname,
            'Target Clips': d_target_clips,
            'Completed Clips': d_comp,
            'Failed / Skipped': d_fail,
            'Audio Files': d_aud,
            'Visual Folders': d_vis,
            'Progress (%)': f"{pct:.1f}%"
        })
        
    print("\n" + "=" * 85)
    print("                      🏆 OVERALL PREPROCESSING SUMMARY 🏆")
    print("=" * 85)
    
    df_summary = pd.DataFrame(summary_rows)
    print(df_summary.to_string(index=False))
    
    total_pct = (overall_completed / GRAND_TOTAL_DATASET) * 100
    print("-" * 85)
    print(f"TOTAL PROCESSED CLIPS : {overall_completed:,} / {GRAND_TOTAL_DATASET:,} ({total_pct:.2f}%)")
    print(f"TOTAL FAILED / SKIPPED: {overall_failed:,} clips")
    print(f"TOTAL AUDIO TENSORS   : {overall_audios:,} .npy files")
    print(f"TOTAL VISUAL FOLDERS  : {overall_visuals:,} folders (with 8 frames each)")
    print("=" * 85)
    
    if total_pct >= 95.0:
        print("🎉 [READY FOR TRAINING] Mahigit 95%+ na ang tapos! Handa na para sa Model Training!")
    else:
        remaining = GRAND_TOTAL_DATASET - overall_completed
        print(f"⏳ [IN PROGRESS] May {remaining:,} clips pa ang pinoproseso ng mga accounts.")